
# **SEMESTER PROJECT DELIVERABLE 2 (ML ENGINE)**

## **Group:** 4

## **Members:**

##### 191231 - Patrick Mungai

##### 192082 - DavidTelios Muriithi


## **Model:** Logistic Regression

## **Task:** Resume to job-category recommendation (multi-class text classification used for top-k ranking).

## **Dataset:** Kaggle Resume Dataset - [https://www.kaggle.com/datasets/snehaanbhawal/resume-dataset]()

In [2]:
import re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
matches = list(Path(".").rglob("Resume.csv"))
if not matches:
    raise FileNotFoundError("Resume.csv not found. Download the Kaggle Resume Dataset (link in README), unzip it, and place Resume.csv inside the 'Resume' folder.")
CSV_PATH = matches[0]
print("Using:", CSV_PATH)

Using: Resume\Resume.csv


In [3]:
raw = pd.read_csv(CSV_PATH)
print("Shape:", raw.shape)
print(raw.dtypes, "\n")
print("Missing values:\n", raw.isnull().sum(), "\n")
print("Number of classes:", raw["Category"].nunique())
raw.head(3)

Shape: (2484, 4)
ID             int64
Resume_str       str
Resume_html      str
Category         str
dtype: object 

Missing values:
 ID             0
Resume_str     0
Resume_html    0
Category       0
dtype: int64 

Number of classes: 24


,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [4]:
df = raw.copy()
text = df["Resume_str"].fillna("")

print("Empty/near-empty (<=200 chars):", (text.str.strip().str.len()<= 200).sum())
print("Exact duplicate texts:", text.duplicated().sum())
print("Contain 'Company Name' placeholder:", text.str.contains("Company Name").sum())
print("Contain encoding junk:", text.str.contains("Â|ï¼|â€|Ã", regex = True).sum())
print("Shortest lengths:", sorted(text.str.len())[:5])
print("Longest length:", text.str.len().max())
print("\nHow a resume opens:\n", text.iloc[0][:150])

Empty/near-empty (<=200 chars): 1
Exact duplicate texts: 2
Contain 'Company Name' placeholder: 2483
Contain encoding junk: 3
Shortest lengths: [21, 881, 985, 995, 1033]
Longest length: 38842

How a resume opens:
          HR ADMINISTRATOR/MARKETING ASSOCIATE

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in H


In [5]:
df = raw.copy()
log = []
def step(name, before, after):
    log.append({"step": name, "rows_before": before, "rows_after": after})

# Drop the redundant HTML column
df = df.drop(columns=["Resume_html"])

# Drop empty resumes
n = len(df)
df["Resume_str"] = df["Resume_str"].fillna("")
df = df[df["Resume_str"].str.strip().str.len() > 200]
step(" Drop empty (<=200 chars)", n, len(df))

# Drop duplicate texts
n = len(df)
df = df.drop_duplicates(subset="Resume_str", keep="first")
step("Drop duplicates", n, len(df))

# Keeping the original for EDA
df["raw_text"] = df["Resume_str"]
df = df.reset_index(drop = True)
pd.DataFrame(log)

,step,rows_before,rows_after
0,Drop empty (<=200 chars),2484,2483
1,Drop duplicates,2483,2481


In [8]:
def fix_encoding_junk(text):
    for bad in ["Â", "ï¼", "â€“", "â€™", "Ã"]:
        text = text.replace(bad, " ")
    return text

def strip_title(text):
    m = re.match(r"^\s*(?:[A-Z0-9/][A-Z0-9/&,\-\.\(\)']*\s+)+", text)
    s = text[m.end():] if m else text
    return s if len(s.strip()) > 200 else text

def clean(text):
    text = strip_title(text)
    text = fix_encoding_junk(text)
    text = text.replace("Company Name", " ").replace("City, State", " ")
    text = re.sub(r"[^A-Za-z+#\s]", " ", text) # Keeps letters, + and #
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text

df["text"] = df["Resume_str"].apply(clean)
n = len(df)
df = df[df["text"].str.len() > 100].reset_index(drop = True)
step(" Drop resumes empty after cleaning (<=100 chars)", n, len(df))
pd.DataFrame(log)


,step,rows_before,rows_after
0,Drop empty (<=200 chars),2484,2483
1,Drop duplicates,2483,2481
2,Drop resumes empty after cleaning (<=100 chars),2481,2481
3,Drop resumes empty after cleaning (<=100 chars),2481,2481
4,Drop resumes empty after cleaning (<=100 chars),2481,2481
